# Can we predict failure within 90 days of an inspection?

Target: `failed_90d`. Plan: inspect data, check for leakage, build baselines, evaluate with a **time-based** split (train on earlier inspections, test on the latest), and report honestly.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from sklearn.inspection import permutation_importance
pd.set_option('display.width', 140)
df = pd.read_csv('data.csv', parse_dates=['inspected_at']).sort_values('inspected_at').reset_index(drop=True)
print(df.shape, 'assets:', df.asset_tag.nunique(), 'missing:', df.isna().sum().sum(), 'dupes:', df.duplicated().sum())
print('failure rate:', df.failed_90d.mean().round(4), '| range', df.inspected_at.min(), '->', df.inspected_at.max())
df.describe().T

## Leakage check: `parts_replaced`

The dictionary says it counts parts replaced on the work order *that closed this inspection record*. That is recorded after the inspection and plausibly after a failure (repairs). Let's see.

In [ ]:
df.groupby('parts_replaced').failed_90d.agg(['mean', 'size'])

Every record with 2 or more parts replaced failed, and none with 0 did. That near-perfect separation is the signature of the outcome leaking into the feature (parts get replaced *because* the asset failed), not of a usable predictor. It would not be known at inspection time, so **I exclude it** from the models.

## Exploration
Each asset appears once, so there is no asset-level leakage across splits. The failure rate varies a lot over time, which matters for the split.

In [ ]:
print(df.groupby('site').failed_90d.agg(['mean', 'size']))
print(df.groupby(df.inspected_at.dt.to_period('Q')).failed_90d.agg(['mean', 'size']))
feats = ['vibration_rms', 'bearing_temp_c', 'ambient_temp_c', 'meter_hours', 'load_pct']
fig, ax = plt.subplots(1, 5, figsize=(18, 3))
for a, f in zip(ax, feats):
    b = pd.qcut(df[f], 10, duplicates='drop')
    r = df.groupby(b, observed=True).failed_90d.mean()
    a.plot(range(len(r)), r.values, marker='o'); a.set_title(f); a.set_xlabel('decile'); a.set_ylim(0)
ax[0].set_ylabel('failure rate'); plt.tight_layout(); plt.show()
df[feats + ['failed_90d']].corr()['failed_90d']

## Time-based split
Train on the first 70% of inspections by date, validate on the next 15%, test on the last 15%.

In [ ]:
n = len(df); i1, i2 = int(n * .7), int(n * .85)
tr, va, te = df.iloc[:i1], df.iloc[i1:i2], df.iloc[i2:]
for nm, s in [('train', tr), ('val', va), ('test', te)]:
    print(nm, len(s), s.inspected_at.min().date(), s.inspected_at.max().date(), 'rate', round(s.failed_90d.mean(), 3))
num = feats; cat = ['site']
def prep(scale):
    return ColumnTransformer([('n', StandardScaler() if scale else 'passthrough', num),
                              ('c', OneHotEncoder(handle_unknown='ignore'), cat)])
models = {
    'logistic': make_pipeline(prep(True), LogisticRegression(max_iter=1000)),
    'grad_boost': make_pipeline(prep(False), HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)),
}
def ev(m, s):
    p = m.predict_proba(s[num + cat])[:, 1]
    return roc_auc_score(s.failed_90d, p), average_precision_score(s.failed_90d, p), p
rows = []
for k, m in models.items():
    m.fit(tr[num + cat], tr.failed_90d)
    for nm, s in [('val', va), ('test', te)]:
        a, ap, _ = ev(m, s); rows.append((k, nm, a, ap, s.failed_90d.mean()))
pd.DataFrame(rows, columns=['model', 'split', 'roc_auc', 'avg_precision', 'base_rate'])

Average precision should be read against the base rate (a random model scores about the base rate).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for k, m in models.items():
    _, _, p = ev(m, te)
    f, t, _ = roc_curve(te.failed_90d, p); ax[0].plot(f, t, label=k)
    pr, rc, _ = precision_recall_curve(te.failed_90d, p); ax[1].plot(rc, pr, label=k)
ax[0].plot([0, 1], [0, 1], 'k--'); ax[0].set(title='ROC (test)', xlabel='FPR', ylabel='TPR')
ax[1].axhline(te.failed_90d.mean(), color='k', ls='--'); ax[1].set(title='Precision-recall (test)', xlabel='recall', ylabel='precision')
ax[0].legend(); plt.tight_layout(); plt.show()

## Which features matter, and how certain is the result?

In [ ]:
m = models['grad_boost']
pi = permutation_importance(m, te[num + cat], te.failed_90d, scoring='roc_auc', n_repeats=20, random_state=0)
print(pd.Series(pi.importances_mean, index=num + cat).sort_values(ascending=False))
_, _, p = ev(m, te); y = te.failed_90d.values; rng = np.random.default_rng(0); b = []
for _ in range(1000):
    ix = rng.integers(0, len(y), len(y))
    if y[ix].min() != y[ix].max(): b.append(roc_auc_score(y[ix], p[ix]))
print('test AUC 95% CI:', np.percentile(b, [2.5, 97.5]).round(3))
fl = p >= np.quantile(p, .9)
print('Top 10%% flagged: precision %.2f vs base rate %.2f; captures %.0f%% of failures' % (y[fl].mean(), y.mean(), 100 * y[fl].sum() / y.sum()))

## Conclusion (numbers from a run of the cells above)

- **`parts_replaced` is leakage** and was excluded. Including it would give a near-perfect but meaningless score.
- With legitimate inspection-time features, there is **modest signal**: gradient boosting reaches ROC AUC about 0.75 on validation and 0.78 on test (bootstrap 95% CI roughly 0.69 to 0.85); logistic regression is weaker (0.68 / 0.64), so the relationship is non-linear. Load, bearing temperature and vibration carry most of the signal.
- **Practical usefulness is limited.** On the test period, flagging the top 10% riskiest inspections gives precision of about 6% (base rate 2.7%) and catches about 21% of failures. Average precision is 0.07 versus a 0.027 base rate.
- **Caveats:** the failure rate swings from 27% (2024 Q2-Q3) to 3% (2025 Q4), so the test period is very different from training. The late drop may partly be right-censoring (inspections within 90 days of the end of the data cannot yet show a failure); confirm the data cut-off date. Test positives are few (about 24), so metrics are noisy. One row per asset means no per-asset history features. Single time split only.
- **Answer:** yes, somewhat better than chance, but not yet reliable enough to act on alone. Next steps: confirm censoring, add asset history (trends across inspections), and re-evaluate with rolling-origin splits.